# ME-TST / GLSD 环境精简版

本 notebook 只保留原 `GLSD+ME-TST.ipynb` 中**最终成功建立运行环境所需的步骤**，
删除前面反复失败的 Python 3.10 / requirements / conda 尝试，以及错误的 dlib patch。

用途：
- 恢复 ME-TST 原工程；
- 建立原实验最终使用的 Python 3.8 + PyTorch 2.1.0 环境；
- 安装原 notebook 最终成功的依赖版本；
- 验证 `Utils`、`mamba_ssm`、`causal_conv1d` 可导入。

**不会运行 backbone、不会生成 response、不会修改 Drive 中 locked evidence。**

建议：Colab 运行时选择 GPU，然后从上到下依次执行。


## 1. 挂载 Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("DRIVE_MOUNT = PASS")


## 2. 干净恢复 ME-TST 仓库

这里会删除 `/content/ME-TST` 的**临时 Colab 目录**后重新 clone。
不会删除 Google Drive 中的任何文件。
这样可以避免之前残留的不完整仓库导致 `Utils` 缺失。


In [ ]:
%cd /content
!rm -rf /content/ME-TST
!git clone https://github.com/zizheng-guo/ME-TST.git /content/ME-TST

from pathlib import Path

repo = Path("/content/ME-TST")
assert (repo / "Utils").exists(), "ME-TST/Utils not found after clean clone"
assert (repo / "Utils/mean_average_precision_str").exists(), "mean_average_precision_str folder missing"

print("METST_REPO_CLONE = PASS")
print("repo =", repo)


## 3. 安装 micromamba


In [ ]:
%cd /content
!rm -rf /content/bin
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj

from pathlib import Path
mamba = Path("/content/bin/micromamba")
assert mamba.exists(), "micromamba binary missing"
print("MICROMAMBA_INSTALL = PASS")
print("micromamba =", mamba)


## 4. 创建最终成功的 ME-TST 环境

对应原 notebook 最终采用的环境：
- Python 3.8
- PyTorch 2.1.0
- torchvision 0.16.0
- torchaudio 2.1.0
- CUDA 12.1


In [ ]:
!/content/bin/micromamba env remove -n ME-TST -y || true

!/content/bin/micromamba create -y -n ME-TST   -c pytorch -c nvidia -c conda-forge   python=3.8   pytorch=2.1.0   torchvision=0.16.0   torchaudio=2.1.0   pytorch-cuda=12.1   pip

!/content/bin/micromamba run -n ME-TST python --version


## 5. 验证 PyTorch / CUDA


In [ ]:
!/content/bin/micromamba run -n ME-TST python -c "import torch; print('torch =', torch.__version__); print('torch cuda =', torch.version.cuda); print('cuda available =', torch.cuda.is_available())"


## 6. 安装原 notebook 最终成功的基础依赖


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install   natsort==7.1.1   einops==0.8.0   tqdm==4.66.4   pandas==1.1.3   scikit-learn==0.23.2   scipy==1.5.4   xlrd==1.2.0   opencv-python-headless==4.5.1.48

!/content/bin/micromamba run -n ME-TST pip install --no-deps timm==1.0.7


## 7. 验证基础依赖与 ME-TST Utils


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import torch,natsort,einops,timm,pandas,sklearn,scipy,cv2; print('torch =',torch.__version__); print('cuda =',torch.cuda.is_available()); print('BASIC_OK')"

!/content/bin/micromamba run -n ME-TST python -c "from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('METST_UTILS_IMPORT = PASS')"


## 8. 安装 causal-conv1d


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     causal-conv1d==1.4.0     --no-build-isolation


## 9. 安装 mamba-ssm


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     mamba-ssm==1.2.0.post1     --no-build-isolation


## 10. 最终环境检查


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import sys, torch, pandas, mamba_ssm, causal_conv1d; from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('Python:', sys.version); print('torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('pandas:', pandas.__version__); print('mamba_ssm: OK'); print('causal_conv1d: OK'); print('Utils metric: OK'); print('ME_TST_ENVIRONMENT = PASS')"


## 下一步：S_alpha 加权融合超参数消融

主实验与模块消融固定 alpha=0.5；本 notebook 追加 alpha={0,0.3,0.5,0.7,1} 的独立 nested LOSO 敏感性实验。
每个 alpha 使用相同的 a0/rho/tau 搜索网格和一对一评价协议；alpha=0/1 已由 L-only/G-only 模块消融覆盖。
不要重新生成 official response，只读取 Drive 中已经锁定的 frozen response。

In [ ]:
from google.colab import drive, files
from pathlib import Path
import io, zipfile, subprocess, sys

SETTING = "sammlv"  # 第二次运行 CAS(ME)3 时改成 "casme3"
PROBE_ONLY = True  # 先检查；通过后改为 False 再运行 cell 24
drive.mount('/content/drive')
uploaded = files.upload()  # 上传 metst_weighted_alpha_hparam_ablation.zip
name = 'metst_weighted_alpha_hparam_ablation.zip'
assert name in uploaded, '请上传 metst_weighted_alpha_hparam_ablation.zip'
task_dir = Path('/content/glsd_weighted_alpha_hparam_ablation')
task_dir.mkdir(exist_ok=True)
allowed = {'run_alpha_hparam_ablation.py', 'run_full_fusion_tuning.py', 'run_fusion_screening.py', 'one_to_one_evaluator.py', 'official_response_component_ablation.py', 'test_alpha_hparam_ablation.py'}
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    assert set(archive.namelist()) == allowed, f'脚本包内容不匹配: {sorted(archive.namelist())}'
    assert archive.testzip() is None, 'ZIP 文件损坏'
    archive.extractall(task_dir)
print('加权融合 alpha 消融脚本已准备:', task_dir)

In [ ]:
import os
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join([str(Path('/content/ME-TST')), str(task_dir), env.get('PYTHONPATH', '')])
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(task_dir), '-p', 'test_alpha_hparam_ablation.py', '-v'], env=env, cwd='/content/ME-TST', check=True)
print('ALPHA_HPARAM_UNIT_TESTS = PASS')
subprocess.run([sys.executable, str(task_dir / 'run_alpha_hparam_ablation.py'), '--setting', SETTING, '--check-inputs'], env=env, cwd='/content/ME-TST', check=True)

In [ ]:
from datetime import datetime, timezone
import json, os, shutil
run_tag = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
output_base = Path('/content/drive/MyDrive/GLSD_WEIGHTED_ALPHA_HPARAM_ABLATION')
output_base.mkdir(parents=True, exist_ok=True)
alpha_output = output_base / (SETTING + '_' + run_tag)
alpha_log = output_base / (SETTING + '_' + run_tag + '.log')
alphas = '0,0.3,0.5,0.7,1'  # unified alpha ablation: endpoints + intermediate weights
bootstrap = """
import runpy, sys
try:
    import pandas as pd
    if not hasattr(pd.DataFrame, 'append'):
        def dataframe_append(self, other, ignore_index=False, verify_integrity=False, sort=False):
            return pd.concat([self, other], ignore_index=ignore_index, verify_integrity=verify_integrity, sort=sort)
        pd.DataFrame.append = dataframe_append
except ImportError:
    pass
sys.argv = sys.argv[1:]
runpy.run_path(sys.argv[0], run_name='__main__')
"""
command = [sys.executable, '-u', '-c', bootstrap, str(task_dir / 'run_alpha_hparam_ablation.py'), '--setting', SETTING, '--alphas', alphas, '--output', str(alpha_output)]
if PROBE_ONLY:
    command.append('--probe-only')
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join([str(Path('/content/ME-TST')), str(task_dir), env.get('PYTHONPATH', '')])
print('输出目录:', alpha_output)
print('完整日志:', alpha_log)
with alpha_log.open('x', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd='/content/ME-TST')
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        raise
assert returncode == 0, f'运行失败，请查看日志: {alpha_log}'
assert (alpha_output / 'completion.json').is_file(), '没有完成标志'
archive = shutil.make_archive('/content/' + alpha_output.name + '_results', 'zip', root_dir=str(alpha_output.parent), base_dir=alpha_output.name)
print('结果 ZIP:', archive)

In [ ]:
from pathlib import Path
import json
import pandas as pd

alpha_output = Path(alpha_output)
completion = json.loads((alpha_output / 'completion.json').read_text(encoding='utf-8'))
assert completion.get('completed') is True, '实验尚未完成，请检查运行日志'
if completion.get('probe_only') is True:
    print('PROBE 已通过；将 PROBE_ONLY=False 后运行实验 cell，才能生成 alpha 汇总表。')
else:
    # 根目录保存跨 setting 的 full 汇总；各 setting 子目录保存 raw/full 明细。
    summary = pd.read_csv(alpha_output / 'alpha_summary_full.csv')
    display(summary[['setting', 'variant', 'alpha', 'TP', 'FP', 'FN', 'F1']])
    display(pd.read_csv(alpha_output / 'alpha_paired_comparisons.csv'))
    print('MAIN_ALPHA_0.5_IS_INCLUDED =',
          summary.groupby('setting')['alpha'].apply(lambda values: 0.5 in set(values)).to_dict())
